# 05 — Reproducible research tables and export

This notebook collects the machine-readable metrics, exact patient split, and quantum experiment metadata into a compact review package. It intentionally does not manufacture missing results.

In [ ]:
from pathlib import Path
import json, pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / "outputs").exists(): ROOT = ROOT.parent
REPORTS = ROOT / "outputs/reports"
SPLIT = REPORTS / "msd_patient_split.json"
if not SPLIT.exists(): raise FileNotFoundError("Run training first; no held-out split has been recorded.")
split = json.loads(SPLIT.read_text(encoding="utf-8"))
print({k: len(split[k]) for k in ("train", "validation", "test")})
print("Patient overlap:", bool(set(split["train"]) & set(split["validation"]) or set(split["train"]) & set(split["test"]) or set(split["validation"]) & set(split["test"])))

In [ ]:
tables = []
for resolution in (16, 32, 240):
    path = REPORTS / f"msd_test_summary_{resolution}.csv"
    if path.exists():
        frame = pd.read_csv(path); frame.insert(1, "resolution", resolution); tables.append(frame)
if not tables: raise FileNotFoundError("No MSD test metrics found.")
combined = pd.concat(tables, ignore_index=True)
combined.to_csv(REPORTS / "research_summary_table.csv", index=False)
combined

In [ ]:
quantum_path = REPORTS / "quantum" / "msd_quantum_study.json"
if quantum_path.exists():
    quantum = json.loads(quantum_path.read_text(encoding="utf-8"))
    print("Quantum records:", len(quantum.get("results", [])))
else:
    print("No real-data quantum study has been run yet. This is intentionally left blank rather than filled with synthetic results.")

## Before submitting a paper

Confirm the exact dataset version and citation, patient-level split, number of patients and slices, training settings, model provenance, all test metrics, circuit resources, noise model, simulator device, and limitations. Replace any missing value with a transparent statement—never infer a result from a demo or placeholder.